# California house prices with NN'26
https://www.kaggle.com/competitions/california-house-prices-with-nn-26/overview <- все файлы там.

Для обработки данных я выбрал модель гребневой регрессии ridge, она будет предсказывать цены.

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

In [3]:
train = pd.read_csv("housing_train.csv")
test = pd.read_csv("housing_test.csv")

Определяем целевой столбец

In [4]:
target_col = 'median_house_value'

In [5]:
train

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-116.46,33.82,6.0,4863.0,920.0,3010.0,828.0,3.9508,104200.0,INLAND
1,-117.04,34.00,21.0,4624.0,852.0,2174.0,812.0,3.5255,132100.0,INLAND
2,-121.03,37.55,32.0,946.0,198.0,624.0,173.0,1.9728,97900.0,INLAND
3,-117.80,33.68,8.0,2032.0,349.0,862.0,340.0,6.9133,274100.0,<1H OCEAN
4,-122.26,37.83,52.0,1656.0,420.0,718.0,382.0,2.6768,182300.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
14443,-118.10,33.91,36.0,726.0,NaN,490.0,130.0,3.6389,167600.0,<1H OCEAN
14444,-117.24,33.37,14.0,4687.0,793.0,2436.0,779.0,4.5391,180900.0,<1H OCEAN
14445,-121.76,37.33,5.0,4153.0,719.0,2435.0,697.0,5.6306,286200.0,<1H OCEAN
14446,-122.44,37.78,44.0,1545.0,334.0,561.0,326.0,3.8750,412500.0,NEAR BAY


Оцениваем датафрейм: координаты, кол-во комнат, возраст жилья, категориальный признак, а также видим, что есть пропуски (14443 строка значение NaN)

Удалим колонку-ответ median_house_value, сделаем целевую переменную с правильными ответами и копию тест выборки, чтобы не обращаться постоянно к загруженному оригинальному.

In [6]:
X_train_raw = train.drop(columns=[target_col])
y_train = train[target_col]
X_test_raw = test.copy()

Преобразуем категориальный признак ocean_proximity в числа 0 и 1. One-Hot Encoding метод. Избегаем мультиколлинеарность.

In [7]:
X_train_encodded = pd.get_dummies(X_train_raw, columns=['ocean_proximity'], drop_first=True)
X_test_encodded = pd.get_dummies(X_test_raw, columns=['ocean_proximity'], drop_first=True)

Синхронизируем столбцы, чтобы количество колонок test и train сошлось. Я заполнил нулями таковые. 

In [8]:
X_train_encodded, X_test_encodded = X_train_encodded.align(X_test_encodded, join='left', axis=1, fill_value=0)

Заполним NaN-значения. Заводим в них медиану.

In [9]:
X_train_clean = X_train_encodded.fillna(X_train_encodded.median())
X_test_clean = X_test_encodded.fillna(X_test_encodded.median())

Масштабируем признаки: сред. знач. = 0, стандартное отклонение = 1

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

Обучаем модель. Ищем закономерность между признаками и целевой стоимостью. В predictions хранятся предсказания для тестовых данных.

In [11]:
model = Ridge()
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)